In [ ]:
from wind_paths import DATA_DIR, MODEL_DIR, RESULT_DIR
from wind_io import open_data_nc
from two_stage_model import Auto_DA_TransUnet_pytorch
from two_stage_io import (rounded_hour_times, read_wind_array, write_wind_nc,
                          checkpoint_sha256, clear_model_memory)

In [ ]:
from pathlib import Path
import gc
import json
import random
import numpy as np
import xarray as xr
import torch

FORECAST_HOURS = 1

import os
DEVICE = os.environ.get('YANGTZE_DEVICE', 'cuda:0')
TIMESTEP = 6
DOWNNUM = 4
COVNUM = 2
TEST_SIZE = 0.2
VALID_SIZE = 0.1
RESIDUAL_SEED = 25
IO_CHUNK = 128


if isinstance(FORECAST_HOURS, bool) or not isinstance(FORECAST_HOURS, int) or not 1 <= FORECAST_HOURS <= 6:
    raise ValueError('FORECAST_HOURS 必须是 1–6 的整数。')
if DEVICE.startswith('cuda'):
    device_index = torch.device(DEVICE).index or 0
    if not torch.cuda.is_available() or device_index >= torch.cuda.device_count():
        raise RuntimeError(f'{DEVICE} 不可用，请核对当前内核和可见 GPU；可在本框修改 DEVICE。')
    torch.cuda.set_device(device_index)

BASE_MODEL_STEM = MODEL_DIR / f'DA_TransUNet_{FORECAST_HOURS}_24year_changjiang'
RESIDUAL_MODEL_STEM = MODEL_DIR / f'DA_TransUNet_{FORECAST_HOURS}_24year_changjiang_residual'
SCALER_PATH = Path(str(RESIDUAL_MODEL_STEM) + '_normalization.npz')
RUN_INFO_PATH = Path(str(RESIDUAL_MODEL_STEM) + '_run.json')
COMPONENTS = {'u': slice(0, 3), 'v': slice(3, 6), 'w': slice(6, 9)}
LEVELS = np.asarray([850, 925, 1000], dtype=np.float32)

def nc_path(kind, component, suffix=''):
    if kind not in ('testy', 'predicty') or component not in COMPONENTS:
        raise ValueError('kind/component 不合法。')
    return RESULT_DIR / f'da_transunet_{kind}_{component}_{FORECAST_HOURS}_24year_changjiang{suffix}.nc'

def seed_residual_training(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

COMMON_KWARGS = dict(
    test_size=TEST_SIZE, valid_size=VALID_SIZE, k_fold=None,
    task_mode='regression', if_bottom_attention_only='yes', if_resnet='yes',
    layer_time=1, baselayer=16, cov_kernelsize=3, cov_strides=1,
    pool_method='maxpooling', pool_kernel_size=2, pool_strides=2,
    ifnormalization='no', normalization_method=None,
    trans_num=1, num_heads=1, dim_feedforward=128, trans_dropout_rate=0.0,
    activate='tanh', nlp_activate='relu', if_last_act='no',
    if_print_model='yes', loss_function='default', optimizer='Adam',
    metrics='Pearsonr', learning_rate=0.0001, epochs=5000, batch_size=500,
    if_early_stopping=100, ifheatmap='no', ifweight='no', ifmute='no',
    device=DEVICE,
)

if not Path(str(BASE_MODEL_STEM) + '.pth').is_file():
    raise FileNotFoundError(f'未找到对应预报时效的已训练模型：{BASE_MODEL_STEM}.pth')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print(f'预报时效：{FORECAST_HOURS} h；设备：{DEVICE}')
print(f'加载：{BASE_MODEL_STEM}.pth')
print(f'残差模型：{RESIDUAL_MODEL_STEM}.pth')
print(f'结果目录：{RESULT_DIR}；第一阶段后缀 _all，第二阶段后缀 _residual')

In [ ]:
import xarray as xr

import numpy as np
sp,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'Surface-pressure.nc'),'sp','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
st,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / '2m-temperature.nc'),'t2m','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
sd,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / '2m-dewpoint-temperature.nc'),'d2m','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
u850,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'U-component-of-wind-850hpa.nc'),'u','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
u925,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'U-component-of-wind-925hpa.nc'),'u','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
u1000,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'U-component-of-wind-1000hpa.nc'),'u','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
v850,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'V-component-of-wind-850hpa.nc'),'v','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
v925,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'V-component-of-wind-925hpa.nc'),'v','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
v1000,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'V-component-of-wind-1000hpa.nc'),'v','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
w850,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'Vertical-velocity-850hpa.nc'),'w','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
w925,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'Vertical-velocity-925hpa.nc'),'w','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
w1000,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'Vertical-velocity-1000hpa.nc'),'w','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
w850=w850*-0.1
w925=w925*-0.1
w1000=w1000*-0.1

In [ ]:
h = FORECAST_HOURS
vx = np.zeros((3, u850.shape[0]-h, u850.shape[1], u850.shape[2]), dtype='float32')
vy = np.zeros((9, u850.shape[0]-h, u850.shape[1], u850.shape[2]), dtype='float32')
vx[0, :, :] = sp[:-h, :, :]
vx[1, :, :] = st[:-h, :, :]
vx[2, :, :] = sd[:-h, :, :]
vy[0, :, :, :] = u850[h:, :, :]
vy[1, :, :, :] = u925[h:, :, :]
vy[2, :, :, :] = u1000[h:, :, :]
vy[3, :, :, :] = v850[h:, :, :]
vy[4, :, :, :] = v925[h:, :, :]
vy[5, :, :, :] = v1000[h:, :, :]
vy[6, :, :, :] = w850[h:, :, :]
vy[7, :, :, :] = w925[h:, :, :]
vy[8, :, :, :] = w1000[h:, :, :]
vx = vx.transpose(1, 2, 3, 0)
vy = vy.transpose(1, 2, 3, 0)

RAW_TIMES = rounded_hour_times(times)
if np.any(np.diff(RAW_TIMES) != np.timedelta64(1, 'h')):
    raise ValueError('源数据不是连续逐小时记录，不能直接以数组位移表示预报小时数。')
if len(RAW_TIMES)-h != len(vy) or len(vx) != len(vy):
    raise ValueError('输入、目标和时间长度不匹配。')
N_RAW_PAIRS = len(vy)
N_ALL = N_RAW_PAIRS-TIMESTEP+1
TEST_START = int((1-TEST_SIZE)*N_RAW_PAIRS)
TRAIN_STOP = int((1-VALID_SIZE/(1-TEST_SIZE))*TEST_START)
ALL_TIMES = RAW_TIMES[h+TIMESTEP-1:]
TEST_TIMES = ALL_TIMES[TEST_START:]
N_TEST = len(TEST_TIMES)
if not 0 < TRAIN_STOP < TEST_START < N_ALL:
    raise ValueError('样本数不足以使用当前历史窗口和训练/验证/测试比例。')
window_offset = TIMESTEP - 1
train_inputs = vx[:TRAIN_STOP + window_offset]
train_labels = vy[window_offset:TRAIN_STOP + window_offset]
vx_mean = np.nanmean(train_inputs, axis=0)
vx_std = np.nanstd(train_inputs, axis=0)
vy_mean = np.nanmean(train_labels, axis=0)
vy_std = np.nanstd(train_labels, axis=0)
vx_raw = vx
vx = (vx - vx_mean) / vx_std
vy = (vy - vy_mean) / vy_std
del train_inputs, train_labels
lat = np.asarray(lat)
lon = np.asarray(lon)
if vy.shape[1:3] != (len(lat), len(lon)):
    raise ValueError('数据空间轴与纬经度长度不匹配。')
for name, value in [('vx_mean', vx_mean), ('vx_std', vx_std), ('vy_mean', vy_mean), ('vy_std', vy_std)]:
    if not np.isfinite(value).all() or (name.endswith('std') and np.any(value <= 0)):
        raise ValueError(f'{name} 含非有限值或零标准差；请先核对输入数据。')
BASE_CHECKPOINT_SHA256 = checkpoint_sha256(str(BASE_MODEL_STEM)+'.pth')

print('标准化后的 vx/vy：', vx.shape, vy.shape)
print(f'可预测全集 {N_ALL}；训练 {TRAIN_STOP}；验证 {TEST_START-TRAIN_STOP}；测试 {N_TEST}')
print(f'全集：{ALL_TIMES[0]} → {ALL_TIMES[-1]}')
print(f'测试：{TEST_TIMES[0]} → {TEST_TIMES[-1]}')

del sp, st, sd, u850, u925, u1000, v850, v925, v1000, w850, w925, w1000
gc.collect()

In [ ]:
def wind_metadata(component, stage):
    attrs = dict(
        forecast_hours=FORECAST_HOURS, history_steps=TIMESTEP,
        stage=stage, source_model=str(BASE_MODEL_STEM)+'.pth',
        source_model_sha256=BASE_CHECKPOINT_SHA256,
        normalization='training_rows_per_grid_cell_and_channel',
        test_size=TEST_SIZE, valid_size=VALID_SIZE,
    )
    if component == 'w':
        attrs['vertical_velocity_note'] = 'Original notebook convention preserved: saved w = -0.1 * ERA5 omega.'
        attrs['comment'] = 'No further conversion of vertical velocity is applied.'
    return attrs

def check_saved_field(field, expected_times):
    if not np.array_equal(field.time.values, expected_times):
        raise ValueError(f'{field.name}: 保存时次与当前模型的目标时次不一致。')
    for name, expected in [('level', LEVELS), ('latitude', lat), ('longitude', lon)]:
        if field.sizes[name] != len(expected) or not np.allclose(field[name].values, expected, rtol=0, atol=1e-5):
            raise ValueError(f'{field.name}: {name} 坐标不匹配。')

def export_baseline_all():
    outputs = model = prediction = truth = None
    try:
        print('第一阶段：加载已有模型，只推理全集。')
        with torch.no_grad():
            outputs = Auto_DA_TransUnet_pytorch(
                vy, vx, TIMESTEP, DOWNNUM, COVNUM,
                **COMMON_KWARGS,
                if_best_mode='yes', modelpath=str(BASE_MODEL_STEM),
                ifrandom_split='all_test', ifsave='no', savepath=None,
            )
        model, prediction, truth = outputs[:3]
        expected = (N_ALL, 9, len(lat), len(lon))
        if prediction.shape != expected or truth.shape != expected:
            raise ValueError(f'all_test 输出形状不匹配：{prediction.shape}, {truth.shape}; 预期 {expected}')
        for start in range(0, N_ALL, IO_CHUNK):
            stop = min(start+IO_CHUNK, N_ALL)
            expected_truth = vy[TIMESTEP-1+start:TIMESTEP-1+stop].transpose(0, 3, 1, 2)
            np.testing.assert_allclose(truth[start:stop], expected_truth, rtol=1e-6, atol=1e-6)
        for component, channels in COMPONENTS.items():
            mean = vy_mean[:, :, channels].transpose(2, 0, 1)
            std = vy_std[:, :, channels].transpose(2, 0, 1)
            for kind, values in [('testy', truth), ('predicty', prediction)]:
                path = write_wind_nc(
                    nc_path(kind, component, '_all'), f'{kind}_{component}',
                    values[:, channels], ALL_TIMES, lat, lon, LEVELS,
                    mean=mean, std=std, attrs=wind_metadata(component, 'baseline_all'),
                    chunk_size=IO_CHUNK,
                )
                print('已保存：', path)
    finally:
        if model is not None:
            clear_model_memory(model)
        outputs = model = prediction = truth = None
        gc.collect()
        clear_model_memory()

export_baseline_all()
del vx, vy
gc.collect()
clear_model_memory()
print('第一阶段完成，模型引用和 CUDA 缓存已释放；原始输入已保留。')

In [ ]:
def prepare_residual_inputs_and_labels():
    offset = TIMESTEP-1
    issue_times = ALL_TIMES-np.timedelta64(FORECAST_HOURS, 'h')
    np.testing.assert_array_equal(issue_times, RAW_TIMES[offset:N_RAW_PAIRS])
    residual_test_start = TEST_START-offset
    residual_train_stop = TRAIN_STOP-offset
    if not 0 < residual_train_stop < residual_test_start < N_ALL-offset:
        raise ValueError('有效样本不足以对齐两阶段的训练、验证和测试边界。')
    inputs = np.empty((N_ALL, len(lat), len(lon), 12), dtype=np.float32)
    labels = np.empty((N_ALL, len(lat), len(lon), 9), dtype=np.float32)
    inputs[:, :, :, :3] = vx_raw[offset:]
    for component, channels in COMPONENTS.items():
        input_channels = slice(3+channels.start, 3+channels.stop)
        with xr.open_dataset(nc_path('testy', component, '_all')) as ds_truth, xr.open_dataset(nc_path('predicty', component, '_all')) as ds_prediction:
            truth = read_wind_array(ds_truth, f'testy_{component}')
            prediction = read_wind_array(ds_prediction, f'predicty_{component}')
            check_saved_field(truth, ALL_TIMES)
            check_saved_field(prediction, ALL_TIMES)
            for start in range(0, N_ALL, IO_CHUNK):
                stop = min(start+IO_CHUNK, N_ALL)
                observed = np.asarray(truth.isel(time=slice(start, stop)).values, dtype=np.float32)
                forecast = np.asarray(prediction.isel(time=slice(start, stop)).values, dtype=np.float32)
                error = observed-forecast
                if not np.isfinite(forecast).all() or not np.isfinite(error).all():
                    raise ValueError(f'{component}: 预测输入或残差标签中出现非有限值。')
                inputs[start:stop, :, :, input_channels] = forecast.transpose(0, 2, 3, 1)
                labels[start:stop, :, :, channels] = error.transpose(0, 2, 3, 1)

    input_stop = residual_train_stop + offset
    surface_inputs = inputs[:, :, :, :3]
    surface_input_mean = np.nanmean(surface_inputs[:input_stop], axis=0)
    surface_input_std_raw = np.nanstd(surface_inputs[:input_stop], axis=0)
    surface_input_std = np.where(surface_input_std_raw <= 1e-12, np.float32(1), surface_input_std_raw)
    prediction_inputs = inputs[:, :, :, 3:]
    prediction_input_mean = np.nanmean(prediction_inputs[:input_stop], axis=0)
    prediction_input_std_raw = np.nanstd(prediction_inputs[:input_stop], axis=0)
    residual_mean = np.nanmean(labels[offset:input_stop], axis=0)
    residual_std_raw = np.nanstd(labels[offset:input_stop], axis=0)
    for name, value in [('prediction_input_mean', prediction_input_mean),
                        ('prediction_input_std_raw', prediction_input_std_raw),
                        ('residual_mean', residual_mean), ('residual_std_raw', residual_std_raw)]:
        if not np.isfinite(value).all():
            raise ValueError(f'{name}: 标准化统计量存在非有限值。')
    prediction_input_std = np.where(prediction_input_std_raw <= 1e-12, np.float32(1), prediction_input_std_raw)
    residual_std = np.where(residual_std_raw <= 1e-12, np.float32(1), residual_std_raw)
    for start in range(0, N_ALL, IO_CHUNK):
        stop = min(start+IO_CHUNK, N_ALL)
        surface_inputs[start:stop] = (surface_inputs[start:stop] - surface_input_mean) / surface_input_std
        prediction_inputs[start:stop] = (prediction_inputs[start:stop]-prediction_input_mean)/prediction_input_std
        labels[start:stop] = (labels[start:stop]-residual_mean)/residual_std

    residual_test_size = 1-(residual_test_start+0.5)/N_ALL
    residual_valid_size = (1-residual_test_size)*(1-(residual_train_stop+0.5)/residual_test_start)
    split = dict(test_size=float(residual_test_size), valid_size=float(residual_valid_size),
                 test_start=int(residual_test_start), train_stop=int(residual_train_stop))
    assert int((1-split['test_size'])*len(labels)) == residual_test_start
    assert int((1-split['valid_size']/(1-split['test_size']))*residual_test_start) == residual_train_stop
    stage2_target_times = ALL_TIMES[offset:]
    np.testing.assert_array_equal(stage2_target_times[residual_test_start:], TEST_TIMES)
    np.testing.assert_array_equal(stage2_target_times[residual_train_stop:residual_test_start],
                                  ALL_TIMES[TRAIN_STOP:TEST_START])
    assert len(labels[offset+residual_test_start:]) == N_TEST

    np.savez_compressed(
        SCALER_PATH,
        vx_mean=vx_mean, vx_std=vx_std, vy_mean=vy_mean, vy_std=vy_std,
        surface_input_mean=surface_input_mean, surface_input_std=surface_input_std,
        prediction_input_mean=prediction_input_mean,
        prediction_input_std=prediction_input_std, prediction_input_std_raw=prediction_input_std_raw,
        residual_mean=residual_mean, residual_std=residual_std, residual_std_raw=residual_std_raw,
        input_scheme='vx_plus_baseline_prediction',
        input_channels=np.asarray(['sp', 'st', 'sd', 'pred_u850', 'pred_u925', 'pred_u1000',
                                   'pred_v850', 'pred_v925', 'pred_v1000', 'pred_w850', 'pred_w925', 'pred_w1000']),
        forecast_hours=FORECAST_HOURS, timestep=TIMESTEP, effective_history_steps=2*TIMESTEP-1,
        test_size=split['test_size'], valid_size=split['valid_size'],
        test_start=split['test_start'], train_stop=split['train_stop'],
        baseline_test_size=TEST_SIZE, baseline_valid_size=VALID_SIZE,
        baseline_test_start=TEST_START, baseline_train_stop=TRAIN_STOP,
        all_times=ALL_TIMES, input_issue_times=issue_times,
        residual_target_times=stage2_target_times, test_times=TEST_TIMES,
        latitude=lat, longitude=lon, level=LEVELS,
        source_model_sha256=BASE_CHECKPOINT_SHA256,
    )
    print('第二阶段输入（原始 3 通道 + 第一阶段预测 9 通道）：', inputs.shape)
    print('残差标签（滑窗前）：', labels.shape)
    print(f'有效历史输入合计 {2*TIMESTEP-1} 个时次；最早舍弃 {offset} 个训练目标。')
    print('第二阶段切分参数：', split)
    print(f'验证时次保持一致；测试集长度仍为 {N_TEST}，目标时次完全一致。')
    print('训练期标准化统计量已保存：', SCALER_PATH)
    return inputs, labels, residual_mean, residual_std, split

vx_residual, vy_residual, residual_mean, residual_std, RESIDUAL_SPLIT = prepare_residual_inputs_and_labels()
RESIDUAL_COMMON_KWARGS = dict(COMMON_KWARGS, test_size=RESIDUAL_SPLIT['test_size'],
                            valid_size=RESIDUAL_SPLIT['valid_size'])
gc.collect()

In [ ]:
def train_and_save_residual():
    outputs = model = prediction = residual_truth = None
    corrected = wind_truth = None
    written_paths = []
    try:
        seed_residual_training(RESIDUAL_SEED)
        print('第二阶段：原始 vx + 第一阶段预测作为 12 通道输入；验证与测试目标时次对齐。')
        outputs = Auto_DA_TransUnet_pytorch(
            vy_residual, vx_residual, TIMESTEP, DOWNNUM, COVNUM,
            **RESIDUAL_COMMON_KWARGS,
            if_best_mode='no', modelpath=None, ifrandom_split='no',
            ifsave='yes', savepath=str(RESIDUAL_MODEL_STEM),
        )
        model, prediction, residual_truth = outputs[:3]
        expected = (N_TEST, 9, len(lat), len(lon))
        if prediction.shape != expected or residual_truth.shape != expected:
            raise ValueError(f'残差测试集形状不匹配：{prediction.shape}, {residual_truth.shape}; 预期 {expected}')
        for start in range(0, N_TEST, IO_CHUNK):
            stop = min(start+IO_CHUNK, N_TEST)
            label_start = TIMESTEP-1+RESIDUAL_SPLIT['test_start']
            expected_truth = vy_residual[label_start+start:label_start+stop].transpose(0, 3, 1, 2)
            np.testing.assert_allclose(residual_truth[start:stop], expected_truth, rtol=1e-6, atol=1e-6)

        clear_model_memory(model)
        outputs = model = residual_truth = None
        gc.collect()
        clear_model_memory()

        for component, channels in COMPONENTS.items():
            mean = residual_mean[:, :, channels].transpose(2, 0, 1)
            std = residual_std[:, :, channels].transpose(2, 0, 1)
            corrected = np.empty((N_TEST, 3, len(lat), len(lon)), dtype=np.float32)
            wind_truth = np.empty_like(corrected)
            with xr.open_dataset(nc_path('testy', component, '_all')) as ds_truth, \
                 xr.open_dataset(nc_path('predicty', component, '_all')) as ds_prediction:
                original_truth = read_wind_array(ds_truth, f'testy_{component}').sel(time=TEST_TIMES)
                baseline_prediction = read_wind_array(ds_prediction, f'predicty_{component}').sel(time=TEST_TIMES)
                check_saved_field(original_truth, TEST_TIMES)
                check_saved_field(baseline_prediction, TEST_TIMES)
                for start in range(0, N_TEST, IO_CHUNK):
                    stop = min(start+IO_CHUNK, N_TEST)
                    residual_physical = np.asarray(prediction[start:stop, channels], dtype=np.float64)*std+mean
                    base = np.asarray(baseline_prediction.isel(time=slice(start, stop)).values, dtype=np.float64)
                    final = base+residual_physical
                    if not np.isfinite(final).all() or np.any(np.abs(final) > np.finfo(np.float32).max):
                        raise ValueError(f'{component}: 修正预测包含非有限值或溢出。')
                    corrected[start:stop] = final.astype(np.float32)
                    wind_truth[start:stop] = original_truth.isel(time=slice(start, stop)).values

            attrs = wind_metadata(component, 'residual_corrected_test')
            attrs.update(
                residual_model=str(RESIDUAL_MODEL_STEM)+'.pth',
                residual_seed=RESIDUAL_SEED, test_count=N_TEST,
                residual_input_scheme='vx_plus_baseline_prediction', residual_input_channels=12,
                effective_history_steps=2*TIMESTEP-1,
                prediction_definition='baseline_prediction + inverse_standardized_residual_prediction',
                truth_definition='original wind truth, not the residual label',
            )
            for kind, values in [('testy', wind_truth), ('predicty', corrected)]:
                path = write_wind_nc(
                    nc_path(kind, component, '_residual'), f'{kind}_{component}',
                    values, TEST_TIMES, lat, lon, LEVELS,
                    attrs=attrs, chunk_size=IO_CHUNK,
                )
                written_paths.append(str(path))
                print('已保存：', path)
            corrected = wind_truth = None
            gc.collect()

        checkpoint = Path(str(RESIDUAL_MODEL_STEM)+'.pth')
        if not checkpoint.exists():
            raise FileNotFoundError(f'残差模型未保存：{checkpoint}')
        info = dict(
            forecast_hours=FORECAST_HOURS, history_steps=TIMESTEP, device=DEVICE,
            residual_seed=RESIDUAL_SEED, baseline_model=str(BASE_MODEL_STEM)+'.pth',
            baseline_sha256=BASE_CHECKPOINT_SHA256,
            residual_model=str(checkpoint), residual_sha256=checkpoint_sha256(checkpoint),
            normalization_file=str(SCALER_PATH),
            normalization_scope='training input and target rows',
            input_scheme='vx_plus_baseline_prediction', input_channels=12,
            input_channel_order=['sp', 'st', 'sd', 'pred_u850', 'pred_u925', 'pred_u1000',
                                 'pred_v850', 'pred_v925', 'pred_v1000', 'pred_w850', 'pred_w925', 'pred_w1000'],
            effective_history_steps=2*TIMESTEP-1,
            input_alignment='raw predictors and baseline forecasts aligned by forecast initialization time',
            baseline_training_predictions='in-sample predictions from the fixed first-stage checkpoint',
            dropped_training_targets=TIMESTEP-1,
            train_count=RESIDUAL_SPLIT['train_stop'],
            valid_count=RESIDUAL_SPLIT['test_start']-RESIDUAL_SPLIT['train_stop'],
            all_count=N_ALL, test_count=N_TEST,
            test_start_time=str(TEST_TIMES[0]), test_end_time=str(TEST_TIMES[-1]),
            test_size=RESIDUAL_SPLIT['test_size'], valid_size=RESIDUAL_SPLIT['valid_size'],
            baseline_test_size=TEST_SIZE, baseline_valid_size=VALID_SIZE,
            outputs=written_paths, training_kwargs=RESIDUAL_COMMON_KWARGS,
        )
        RUN_INFO_PATH.write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding='utf-8')
        print(f'完成：两阶段测试集均为 {N_TEST} 个时次，时间坐标完全一致。')
        print('残差模型：', checkpoint)
        print('运行记录：', RUN_INFO_PATH)
        return info
    finally:
        if model is not None:
            clear_model_memory(model)
        outputs = model = prediction = residual_truth = corrected = wind_truth = None
        gc.collect()
        clear_model_memory()

residual_run_info = train_and_save_residual()